In [0]:
import os
import sys
from sklearn.pipeline import Pipeline
sys.path.insert(0, '/Workspace/Users/hayley.h.luo@gmail.com/beth-dataset-anamoly-detection')
from model_class.feature_builder import FeatureBuilder
from model_class.feature_builder_transformer import FeatureBuilderTransformer
from sklearn.feature_selection import mutual_info_classif
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import SGDOneClassSVM
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import roc_auc_score, roc_curve

In [0]:
# Get the current directory
raw_directory = "/Volumes/workspace/beth_dataset/raw"
    
# Get the source file path
file = f"{raw_directory}/labelled_training_data.csv"
df_train = pd.read_csv(file, header=0)
y_train = df_train['evil']
X_train = df_train.drop(columns=['sus', 'evil'])

# Get the source file path
file = f"{raw_directory}/labelled_testing_data.csv"
df_test = pd.read_csv(file, header=0)
y_test = df_test['evil']
X_test = df_test.drop(columns=['sus', 'evil'])

# Get the source file path
file = f"{raw_directory}/labelled_validation_data.csv"
df_validation = pd.read_csv(file, header=0)
y_validation = df_validation['evil']
X_validation = df_validation.drop(columns=['sus', 'evil'])

In [0]:

def evaluate_model(y_true, y_pred, dataset_type="Dataset"):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # Get unique values and their counts in y_true
    unique_values, counts = np.unique(y_true, return_counts=True)
    print(f"y_true_{dataset_type} Unique values: {unique_values}")
    print(f"y_true_{dataset_type} Counts of each value: {counts}")

    # Get unique values and their counts in y_pred
    unique_values, counts = np.unique(y_pred, return_counts=True)
    print(f"y_pred_{dataset_type} Unique values: {unique_values}")
    print(f"y_pred_{dataset_type} Counts of each value: {counts}")

     # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print("Confusion Matrix:")
    print(cm)
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Inlier", "Outlier"])
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix for {dataset_type}')
    plt.show()

    # ROC AUC Score
    roc_auc = roc_auc_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    fnr = 1-tpr
    tnr = 1-fpr

    print(f"False Positive Rate: {fpr}")
    print(f"True Positive Rate: {tpr}")
    print(f"False Negative Rate: {fnr}")
    print(f"True Negative Rate: {tnr}")


    if dataset_type == "Test Dataset":
        plt.figure()
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'Receiver Operating Characteristic for {dataset_type}')
        plt.legend(loc="lower right")
        plt.savefig(f"{raw_directory}/confusion_matrix/{dataset_type}_confusion_matrix.png")
        plt.show()

In [0]:
def evaluate_model(y_true, y_pred, dataset_type="Dataset"):
    # Returns -1 for outliers and 1 for inliers.
    # change -1 to 0:
    y_pred = np.where(y_pred == -1, 1, 0)

    # Get unique values and their counts in y_true
    unique_values, counts = np.unique(y_true, return_counts=True)
    print(f"y_true_{dataset_type} Unique values: {unique_values}")
    print(f"y_true_{dataset_type} Counts of each value: {counts}")

    # Get unique values and their counts in y_pred
    unique_values, counts = np.unique(y_pred, return_counts=True)
    print(f"y_pred_{dataset_type} Unique values: {unique_values}")
    print(f"y_pred_{dataset_type} Counts of each value: {counts}")

     # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    print(cm)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Inlier", "Outlier"])
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix for {dataset_type}')
    plt.savefig(f"{raw_directory}/confusion_matrix/{dataset_type}_confusion_matrix.png")
    plt.show()

    # ROC AUC Score
    roc_auc = roc_auc_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_pred)

    print(f"False Positive Rate: {fpr}")
    print(f"True Positive Rate: {tpr}")


    if dataset_type == "Test Dataset":
        plt.figure()
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'Receiver Operating Characteristic for {dataset_type}')
        plt.legend(loc="lower right")
        plt.show()
    
    return fpr, tpr, roc_auc

In [0]:
count_cols = [
    "processId_eventId_past_count",
    "processId_past_count",
    "threadId_past_count",
    "eventId_past_count",
    "userId_past_count",
    "parentProcessId_past_count",
    "mountNamespace_past_count",
]

rarity_cols = [
    "processId_eventId_rarity",
    "processId_rarity",
    "threadId_rarity",
    "eventId_rarity",
    "userId_rarity",
    "parentProcessId_rarity",
    "mountNamespace_rarity",
]

scale_cols = [
    "stackAddresses_len",
    "stackAddresses_unique_ratio",
    "child_process_spawn_rate_so_far",
]

binary_cols = [
    "userId_binary",
    "parentUserId_binary",
    "returnValue_is_error",
    "args_has_path",
    "processId_eventId_is_first_seen",
    "processId_is_first_seen",
    "threadId_is_first_seen",
    "eventId_is_first_seen",
    "userId_is_first_seen",
    "parentProcessId_is_first_seen",
    "mountNamespace_is_first_seen"
]

passthrough_cols = [
    "argsNum",
    "returnValue",
]

log_count_pipeline = Pipeline([
    ("log", FunctionTransformer(
        lambda x: np.log1p(np.clip(x, 0, None)),
        feature_names_out="one-to-one",
        validate=False
    )),
    ("scale", RobustScaler()),
])

rarity_pipeline = Pipeline([
    ("scale", RobustScaler()),
])

scale_pipeline = Pipeline([
    ("scale", RobustScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("count", log_count_pipeline, count_cols),
        ("rarity", rarity_pipeline, rarity_cols),
        ("scale_other", scale_pipeline, scale_cols),
        ("binary", "passthrough", binary_cols),
        ("pass", "passthrough", passthrough_cols),
    ]
)

In [0]:
model = Pipeline([
    ("features", FeatureBuilderTransformer(FeatureBuilder(), return_numpy=False)),
    ("scaler", preprocessor),
    ("clf", SGDOneClassSVM(nu=0.05, random_state=42)),
])

In [0]:
print("Fitting model using training data...")
model.fit(X_train)

In [0]:
y_pred_train = (model.predict(X_train) == -1).astype(int)
train_fpr, train_tpr, train_roc = evaluate_model(y_train, y_pred_train, dataset_type="Train Dataset")

y_pred_validation = (model.predict(X_validation) == -1).astype(int)
validation_fpr, validation_tpr, validation_roc = evaluate_model(y_validation, y_pred_validation, dataset_type="Validation Dataset")

y_pred_test = (model.predict(X_test) == -1).astype(int)
test_fpr, test_tpr, test_roc = evaluate_model(y_test, y_pred_test, dataset_type="Test Dataset")